# Hypothesis Testing and Predictive Modelling ( Global CO2 Responsibility)

## Objective
-Statistically test key project hypotheses (trend + relationships)
- Build simple predictive models to support forward-looking insights

## Inputs
-Clean dataset: df_co2_emissions_clean (1990-2024)

## Outputs
- Test results (p-values, correlations)
- Model results (MAE/RMSE/R²)
- Export CSVs for Power BI

In [2]:
import pandas as pd
import numpy as np      
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression   
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline   

In [4]:
# Load the dataset
df_co2_emissions = pd.read_csv('../DataSet/Cleaned/CO2_Emissions_Cleaned.csv')
# Displaying the first few rows of the dataset
df_co2_emissions.head()



,country,iso_code,year,co2,co2_per_capita,gdp,population,cumulative_co2
0,Afghanistan,AFG,1990,2.024326,0.168054,1.306598e+10,12045664.0,58.603493
1,Afghanistan,AFG,1991,1.914301,0.156411,1.204736e+10,12238879.0,60.517792
2,Afghanistan,AFG,1992,1.482054,0.111609,1.267754e+10,13278983.0,61.999847
3,Afghanistan,AFG,1993,1.486943,0.099506,9.834582e+09,14943175.0,63.486794
4,Afghanistan,AFG,1994,1.453829,0.089462,7.919856e+09,16250800.0,64.940620


In [6]:
# Checking datset information
df_co2_emissions.info() 
df_co2_emissions.describe() 




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7508 entries, 0 to 7507
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         7508 non-null   object 
 1   iso_code        7508 non-null   object 
 2   year            7508 non-null   int64  
 3   co2             7508 non-null   float64
 4   co2_per_capita  7442 non-null   float64
 5   gdp             5409 non-null   float64
 6   population      7442 non-null   float64
 7   cumulative_co2  7508 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 469.4+ KB


,year,co2,co2_per_capita,gdp,population,cumulative_co2
count,7508.000000,7508.000000,7442.000000,5.409000e+03,7.442000e+03,7508.000000
mean,2007.035828,137.544464,5.022831,4.895098e+11,3.179892e+07,5822.072674
std,10.083330,675.286457,7.795041,1.705000e+12,1.262322e+08,27988.146188
min,1990.000000,0.000000,0.000000,2.571720e+08,1.776000e+03,0.000000
25%,1998.000000,0.852971,0.714055,1.886576e+10,7.567225e+05,23.068187
50%,2007.000000,7.172069,2.744757,6.111433e+10,5.713997e+06,227.102509
75%,2016.000000,53.937630,6.832723,2.910000e+11,2.045923e+07,1978.472718
max,2024.000000,12289.037110,364.790833,2.700000e+13,1.450936e+09,434866.562500


In [7]:
# Identify latest year in the dataset
latest_year = df_co2_emissions['year'].max()

#Latest year dataset
df_latest = df_co2_emissions[df_co2_emissions['year'] == latest_year].copy()

# Global yearly totals
global_year = (df_co2_emissions.groupby('year', as_index=False)['co2'].sum())

latest_year, df_latest.shape,  global_year.shape


(2024, (215, 8), (35, 2))

## Hypothesis Testing
Hypothesis 1 - Has Global CO2 increased since 1990?

Ho: Global CO2 emissions have not increased over time
H1: Global CO2 emissions have increased over time


Pearson correlation analysis was used to examine the relationship between time (year) and global CO₂ emissions. This statistical test is appropriate because both variables are continuous and normally distributed at the aggregate level. Pearson’s r measures the strength and direction of a linear relationship, producing a coefficient (r) ranging from -1 to +1

In [9]:
# Testing using Pearson Correlation
r_h1, p_h1 = stats.pearsonr(global_year['year'], global_year['co2'])
print('H1 pearson r:', round(r_h1, 3))
print(' H1 p-value:', p_h1)

H1 pearson r: 0.982
 H1 p-value: 1.8306757151155014e-25


 ## H1 Result and Interpretation 
Pearson correlation analysis revealed a strong positive relationship between year and global CO₂ emissions (r = 0.982, p < 0.05).
This indicates that as time progresses, global emissions increase significantly. The strength of the correlation suggests that the rise in emissions is not random but follows a consistent upward trajectory.
From an analytical perspective, this trend reflects the long-term expansion of industrial activity, fossil fuel dependence, and economic growth worldwide. Although short-term declines are visible, such as the temporary drop around 2020,  the overall pattern demonstrates sustained growth in atmospheric carbon output.

## Conclusion
The null hypothesis is therefore rejected. There is sufficient statistical evidence to conclude that global CO₂ emissions have increased significantly since 1990. This finding reinforces the urgency of climate mitigation policies, as historical trends show continued escalation rather than stabilisation.

## Hypothesis 2
H2:Countries with larger populations produce higher total CO2

Ho (Null Hypothesis): There is no statistically significant relationship between a country's GDP and its total CO2 emissions
H1(Alternative Hypothesis) : There is a statistically significant relationship between GDP and total CO2 emissions


Pearson correlation was selected because:
Both variables (GDP and CO₂ emissions) are continuous numerical variables.
The objective is to measure the strength and direction of the linear relationship.
The dataset is sufficiently large (213 countries), making Pearson appropriate and reliable.


In [10]:
# Clean subset for H2
df_h2 = df_latest.dropna(subset=['population', 'co2']).copy()
# Testing using Pearson Correlation
r_h2, p_h2 = stats.pearsonr(df_h2['population'], df_h2['co2'])
print('H2 pearson r:', round(r_h2, 3))
print('H2 p-value:', p_h2)  
print ('Countries used', df_h2['country'].nunique())

H2 pearson r: 0.824
H2 p-value: 5.786672971691484e-54
Countries used 213


## Hypothesis 2 Interpretation
The Pearson correlation coefficient of 0.824 indicates a strong positive relationship between GDP and total CO₂ emissions.
This means that:
Countries with larger economies tend to produce higher total emissions.
Economic output and industrial activity are closely linked to carbon production.
As GDP increases, emissions generally increase as well.
The p-value is far below the standard significance level of 0.05, meaning the result is statistically significant.

Therefore, the null hypothesis is rejected.



# Hypothesis 2 Conclusion 
The hypothesis test confirms a strong and statistically significant relationship between GDP and CO₂ emissions.

While population explains emission scale, GDP reflects the economic intensity behind carbon output. Together, both factors provide a deeper understanding of global emissions responsibility.

## Hypothesis 3
H3: Developed vs Developing : Per-Capita CO2 Emissions

Ho ( Null Hypothesis) : There is no statistically significant difference in average CO2 emissions per capita between developed and developing countries.

H1 ( Alternative Hypothesis): There is a statistically significant difference in average CO2 emissions per capita between developed and developing countries.

Statistical Test Used

An independent samples t-test (Welch’s t-test) was conducted to compare mean per-capita CO2 emissions between developed and developing countries.
This test was selected because:

Two independent groups were being compared.

The sample sizes were unequal.

Variance between groups could differ.

Welch’s t-test is more robust under these conditions.

In [ ]:
# Creating a mapping of countries to their development status
development_status_mapping = {
    'Qatar': 'Developed',
    'Kuwait': 'Developed',  
    'Brunei': 'Developed',
    'Bahrain': 'Developed',
    'United Arab Emirates': 'Developed',
    'New Caledonia': 'Developed',
    'Saudi Arabia': 'Developing',
    'Oman': 'Developing',
    'Trinidad and Tobago': 'Developing',
    'Sint Maarten (Dutch part)': 'Developing'}




In [13]:
# Adding development status to the dataset
h3_df = df_latest[['country', 'co2_per_capita']].copy()
# Add development status to the dataset
h3_df['Development_Status'] = h3_df['country'].map(development_status_mapping)
# Kepp only the 10 mapped countries and  remove missing per-capita values
h3_df = h3_df.dropna(subset=['Development_Status', 'co2_per_capita'])
h3_df


,country,co2_per_capita,Development_Status
559,Bahrain,24.270082,Developed
1084,Brunei,26.046202,Developed
3627,Kuwait,26.247530,Developed
4812,New Caledonia,18.064400,Developed
5127,Oman,15.651107,Developing
5512,Qatar,41.271179,Developed
5897,Saudi Arabia,20.379194,Developing
6107,Sint Maarten (Dutch part),16.546274,Developing
6842,Trinidad and Tobago,22.931944,Developing
7122,United Arab Emirates,20.131075,Developed


In [14]:
# Spliting the groups
developed = h3_df[h3_df['Development_Status'] == 'Developed']['co2_per_capita']
developing = h3_df[h3_df['Development_Status'] == 'Developing']['co2_per_capita']
print('Developed count:', len(developed))
print('Developing count:', len(developing))

Developed count: 6
Developing count: 4


In [19]:
# Run t-test
t_stat, p_value = stats.ttest_ind(developed, developing, equal_var=False)
print('T-statistic:', t_stat)
print('P-value:', p_value)

T-statistic: 1.904962196125825
P-value: 0.09773905692158716


## Hypothesis 3 Interpretation
The independent samples t-test produced a p-value of 0.0977, which is above the standard 0.05 significance level. Therefore, the null hypothesis cannot be rejected.

Although developed countries show a higher average per-capita CO₂ emission compared to developing countries, this difference is not statistically significant within the selected sample.

This suggests that, among the top per-capita emitters analysed, development status alone does not fully explain variation in emissions intensity.


## Hypothesis 3 Conclusion
The lack of statistical significance may be influenced by the limited sample size and the fact that the dataset focuses only on countries with exceptionally high per-capita emissions. A broader global comparison may yield stronger statistical separation between development groups.